# Mastra.ai — Agentes TypeScript: simulación y conceptos en Python

Dado que Mastra es un framework TypeScript, este notebook simula sus patrones
en Python para entender los conceptos antes de implementarlos en Node.js.

In [ ]:
import anthropic, os, json, asyncio
from dataclasses import dataclass, field
from typing import Any, Callable
from datetime import datetime

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('Simulador de Mastra listo')

## 1. Concepto: Agente de Mastra

In [ ]:
# Equivalencia Python de un agente Mastra
# En TypeScript:
#   const agente = new Agent({ name, instructions, model, tools })
#   const resp = await agente.generate('texto')

@dataclass
class AgenteMastra:
    """Simulación de la API de Agent de Mastra en Python."""
    name: str
    instructions: str
    model: str = MODELO
    tools: dict[str, Callable] = field(default_factory=dict)
    memory: list[dict] = field(default_factory=list)  # thread memory

    def generate(self, mensaje: str, thread_id: str | None = None) -> dict:
        """Equivalente a agente.generate() de Mastra."""
        # Recuperar historial del thread
        historial = [m for m in self.memory if m.get('thread_id') == thread_id] if thread_id else []
        mensajes_api = [{'role': m['role'], 'content': m['content']} for m in historial]
        mensajes_api.append({'role': 'user', 'content': mensaje})

        resp = client.messages.create(
            model=self.model, max_tokens=512,
            system=self.instructions,
            messages=mensajes_api,
        )
        texto = resp.content[0].text

        # Guardar en memoria si hay thread_id
        if thread_id:
            self.memory.append({'thread_id': thread_id, 'role': 'user', 'content': mensaje})
            self.memory.append({'thread_id': thread_id, 'role': 'assistant', 'content': texto})

        return {
            'text': texto,
            'usage': {'promptTokens': resp.usage.input_tokens, 'completionTokens': resp.usage.output_tokens},
        }

# Crear un agente al estilo Mastra
agente_legal = AgenteMastra(
    name='Asistente Legal',
    instructions='Eres un experto en contratos legales. Responde de forma clara y concisa en español.',
)

resp = agente_legal.generate('¿Qué es una cláusula arbitral?')
print(f'Respuesta: {resp["text"]}')
print(f'Tokens: {resp["usage"]}')

## 2. Herramientas al estilo Mastra/Zod

In [ ]:
# En Mastra/TypeScript:
#   const tool = createTool({ id, description, inputSchema: z.object({...}), execute })
# En Python simulamos con el esquema de tools de Anthropic + validación manual

from datetime import date, timedelta
from pydantic import BaseModel, Field

class InputCalcularPreaviso(BaseModel):
    fecha_vencimiento: str = Field(description='Fecha ISO 8601, ej: 2026-09-01')
    dias_preaviso: int = Field(ge=1, le=365, description='Días de preaviso requeridos')

class OutputCalcularPreaviso(BaseModel):
    fecha_limite: str
    dias_restantes: int
    urgente: bool
    mensaje: str

def execute_calcular_preaviso(input_data: dict) -> dict:
    datos = InputCalcularPreaviso(**input_data)
    vencimiento = date.fromisoformat(datos.fecha_vencimiento)
    limite = vencimiento - timedelta(days=datos.dias_preaviso)
    hoy = date.today()
    dias_restantes = max(0, (limite - hoy).days)
    output = OutputCalcularPreaviso(
        fecha_limite=limite.isoformat(),
        dias_restantes=dias_restantes,
        urgente=dias_restantes < 30,
        mensaje=f'Preaviso antes del {limite.isoformat()}. {"⚠️ URGENTE" if dias_restantes < 30 else "Con margen"}.',
    )
    return output.model_dump()

# Definir la herramienta en formato Anthropic
TOOL_CALCULAR_PREAVISO = {
    'name': 'calcular_preaviso',
    'description': 'Calcula la fecha límite para enviar un preaviso de no renovación de contrato.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'fecha_vencimiento': {'type': 'string', 'description': 'Fecha ISO 8601'},
            'dias_preaviso': {'type': 'integer', 'minimum': 1, 'maximum': 365},
        },
        'required': ['fecha_vencimiento', 'dias_preaviso'],
    },
}

# Probar la herramienta directamente
resultado_tool = execute_calcular_preaviso({'fecha_vencimiento': '2026-09-01', 'dias_preaviso': 90})
print(json.dumps(resultado_tool, indent=2, ensure_ascii=False))

## 3. Workflow multi-paso (estilo Mastra Steps)

In [ ]:
# Mastra usa .then(step1).then(step2).commit() para workflows
# Aquí simulamos el patrón con una cadena de pasos tipados

from pydantic import BaseModel
from typing import Callable, TypeVar

class StepInput(BaseModel):
    texto_contrato: str

class ClasificacionOutput(BaseModel):
    tipo: str
    riesgo: str
    texto_contrato: str  # pass-through

class PlazosOutput(BaseModel):
    fecha_vencimiento: str | None
    dias_preaviso: int
    renovacion_automatica: bool
    tipo: str  # pass-through
    riesgo: str  # pass-through

class AlertasOutput(BaseModel):
    alertas: list[dict]
    resumen_final: str

def paso_clasificar(input_data: StepInput) -> ClasificacionOutput:
    resp = client.messages.create(
        model=MODELO, max_tokens=100, temperature=0,
        messages=[{'role': 'user', 'content': f'Clasifica este contrato. JSON: {{"tipo": "arrendamiento|servicio|compraventa|otro", "riesgo": "alto|medio|bajo"}}\n{input_data.texto_contrato}'}]
    )
    datos = json.loads(resp.content[0].text)
    return ClasificacionOutput(**datos, texto_contrato=input_data.texto_contrato)

def paso_extraer_plazos(input_data: ClasificacionOutput) -> PlazosOutput:
    resp = client.messages.create(
        model=MODELO, max_tokens=150, temperature=0,
        messages=[{'role': 'user', 'content': f'Extrae plazos de este {input_data.tipo}. JSON: {{"fecha_vencimiento": "YYYY-MM-DD o null", "dias_preaviso": 90, "renovacion_automatica": true}}\n{input_data.texto_contrato}'}]
    )
    datos = json.loads(resp.content[0].text)
    return PlazosOutput(**datos, tipo=input_data.tipo, riesgo=input_data.riesgo)

def paso_generar_alertas(input_data: PlazosOutput) -> AlertasOutput:
    alertas = []
    if input_data.fecha_vencimiento:
        venc = date.fromisoformat(input_data.fecha_vencimiento)
        hoy = date.today()
        for dias, tipo in [(input_data.dias_preaviso, 'preaviso'), (30, 'revision'), (0, 'vencimiento')]:
            fecha_alerta = venc - timedelta(days=dias)
            if fecha_alerta >= hoy:
                alertas.append({'tipo': tipo, 'fecha': fecha_alerta.isoformat(), 'dias_hasta': (fecha_alerta - hoy).days})

    resumen = f'Contrato {input_data.tipo} con riesgo {input_data.riesgo}. {len(alertas)} alertas generadas.'
    return AlertasOutput(alertas=alertas, resumen_final=resumen)

# Ejecutar el workflow
CONTRATO = """
Contrato de servicios cloud entre Cloud SaaS SL y Cliente Final SL.
Duración: 12 meses desde la firma. Vencimiento: 2026-11-01.
Renovación automática salvo preaviso de 90 días. Precio: 2.000€/mes.
"""

step1 = paso_clasificar(StepInput(texto_contrato=CONTRATO))
print(f'Paso 1 — Tipo: {step1.tipo}, Riesgo: {step1.riesgo}')

step2 = paso_extraer_plazos(step1)
print(f'Paso 2 — Vencimiento: {step2.fecha_vencimiento}, Preaviso: {step2.dias_preaviso}d, Auto-renovación: {step2.renovacion_automatica}')

step3 = paso_generar_alertas(step2)
print(f'Paso 3 — {step3.resumen_final}')
for alerta in step3.alertas:
    print(f'  📅 {alerta["tipo"].upper()}: {alerta["fecha"]} (en {alerta["dias_hasta"]} días)')

## 4. Código TypeScript equivalente (referencia)

In [ ]:
CODIGO_TYPESCRIPT = '''
// src/mastra/agents/legal-agent.ts
import { Agent } from "@mastra/core/agent";
import { anthropic } from "@ai-sdk/anthropic";
import { createTool } from "@mastra/core/tools";
import { z } from "zod";

const calcularPreavisoTool = createTool({
  id: "calcular-preaviso",
  description: "Calcula la fecha límite para enviar un preaviso de contrato",
  inputSchema: z.object({
    fechaVencimiento: z.string().describe("Fecha ISO 8601"),
    diasPriorAviso: z.number().min(1).max(365),
  }),
  outputSchema: z.object({
    fechaLimite: z.string(),
    diasRestantes: z.number(),
    urgente: z.boolean(),
  }),
  execute: async ({ context }) => {
    const vencimiento = new Date(context.fechaVencimiento);
    const limite = new Date(vencimiento);
    limite.setDate(limite.getDate() - context.diasPriorAviso);
    const diasRestantes = Math.floor(
      (limite.getTime() - Date.now()) / (1000 * 60 * 60 * 24)
    );
    return {
      fechaLimite: limite.toISOString().split("T")[0],
      diasRestantes: Math.max(0, diasRestantes),
      urgente: diasRestantes < 30,
    };
  },
});

export const legalAgent = new Agent({
  name: "Asistente Legal",
  instructions: "Eres un experto en contratos legales.",
  model: anthropic("claude-haiku-4-5-20251001"),
  tools: { calcularPreaviso: calcularPreavisoTool },
});

// Uso:
const resp = await legalAgent.generate(
  "El contrato vence el 2026-09-01 con 90 días de preaviso. ¿Cuándo es el límite?"
);
console.log(resp.text);
'''

print('Código TypeScript equivalente con Mastra:')
print(CODIGO_TYPESCRIPT)
print('Instalar con: npm install @mastra/core @ai-sdk/anthropic zod')